In [ ]:
pip install sqlalchemy pymysql

In [ ]:
from sqlalchemy import create_engine
import pandas as pd


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
load_dotenv(root / '.env')

username = os.getenv('DB_USERNAME', 'root')
password = quote_plus(os.getenv('DB_PASSWORD', ''))
host = os.getenv('DB_HOST', 'localhost')
port = int(os.getenv('DB_PORT', '3306'))
database = os.getenv('DB_DATABASE', 'crm_sales_db')

connection_url = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
try:
    engine = create_engine(connection_url)
    print('Database connection successful.')
except Exception as exc:
    raise RuntimeError(f'Database connection failed: {exc}') from exc

data_files = {
    root / 'dataset/processed/Accounts_cleaned.csv': 'accounts',
    root / 'dataset/raw/products.csv': 'products',
    root / 'dataset/processed/Sales_pipeline_cleaned.csv': 'sales_pipeline',
    root / 'dataset/raw/sales_teams.csv': 'sales_teams',
    root / 'dataset/raw/data_dictionary.csv': 'data_dictionary',
}

for file_path, table_name in data_files.items():
    if not file_path.exists():
        print(f'File {file_path.name} not found. Skipping...')
        continue
    df = pd.read_csv(file_path)
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    print(f'{table_name} data uploaded successfully.')
